# Training: Instance Segmentation Panoptic Indoor v5

NRP berakhiran 036, digit terakhir 6 (rentang 6–7), variabel eksperimen: augmentasi (tanpa / default / rancangan sendiri).

| Model | Keluarga |
|---|---|
| YOLO11s-seg | One-stage CNN |
| YOLOv8s-seg | One-stage CNN |
| Mask R-CNN R50-FPN v2 | Two-stage CNN |

## 0. Konfigurasi

In [1]:
RF_WORKSPACE, RF_PROJECT, RF_VERSION = "panopticindoor", "panoptic-indoor-segmentation", 5
PREV_OUTPUT = "auto"

MERGE = {
    "ligjt": "light", "liht": "light", "llight": "light",
    "persob": "person",
    "spon": "spoon", "spoom": "spoon",
    "cuo": "cup", "cyo": "cup",
    "palte": "plate",
    "pap": "paper", "paper-merged": "paper",
    "staris": "stairs",
    "stovv": "stove-merged",
    "towl": "towel",
    "scissor": "scissors",
    "phone": "cell phone",
    "ruf": "rug-merged",
    "windo": "window-other",
}
DROP = []

IMGSZ      = 640
MAIN_HOURS = 1.0
SEED = 0

## 1. Setup

In [2]:
import os, glob, json, yaml, shutil, time, datetime, csv, collections, traceback, contextlib, io
assert os.path.exists("/kaggle/working"), "Notebook ini khusus Kaggle."

!pip install -q roboflow ultralytics pycocotools pillow-heif gdown
!pip uninstall -y -q albumentations

import numpy as np, pandas as pd, torch
from kaggle_secrets import UserSecretsClient

ROOT = "/kaggle/working/CV_InstanceSeg"
DATA = "/tmp/data"
RUNS = f"{ROOT}/runs"
os.makedirs(RUNS, exist_ok=True)

def find_output(path):
    if path != "auto":
        return path
    cands = glob.glob("/kaggle/input/**/CV_InstanceSeg", recursive=True)
    return max(cands, key=lambda d: len(glob.glob(f"{d}/runs/*/DONE")), default="")

PREV_OUTPUT = find_output(PREV_OUTPUT)
if PREV_OUTPUT:
    src = f"{PREV_OUTPUT}/CV_InstanceSeg" if os.path.isdir(f"{PREV_OUTPUT}/CV_InstanceSeg") else PREV_OUTPUT
    assert os.path.isdir(f"{src}/runs"), f"Tidak ada folder runs di {src}. Cek path di panel Input."
    shutil.copytree(src, ROOT, dirs_exist_ok=True)
    print("Melanjutkan dari", src, "->", sorted(os.listdir(RUNS)))

assert torch.cuda.is_available(), "GPU tidak aktif: Settings → Accelerator → GPU T4 x2"
DEV = torch.device("cuda")
!nvidia-smi -L

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Melanjutkan dari /kaggle/input/notebooks/apasijann/tugas-viskom-roboflow/CV_InstanceSeg -> ['aug_custom', 'aug_default', 'aug_none', 'mask_rcnn_r50', 'val', 'yolo11s_seg', 'yolov8s_seg']
GPU 0: Tesla T4 (UUID: GPU-b77878cf-5c9e-2bb7-5a44-4bfdfb1aeb60)
GPU 1: Tesla T4 (UUID: GPU-316babc7-9247-ee0c-41f0-6d9edbadabab)


## 2. Fungsi bantu

In [3]:
ERR_LOG = f"{ROOT}/errors.txt"
GPU_LOG = f"{ROOT}/gpu_time_log.csv"

def stage(name, fn, *args, **kw):
    try:
        return fn(*args, **kw)
    except Exception:
        tb = traceback.format_exc()
        print(f"!!! Tahap {name} gagal:\n{tb}")
        with open(ERR_LOG, "a") as f:
            f.write(f"== {name} ({datetime.datetime.now():%Y-%m-%d %H:%M}) ==\n{tb}\n")
        return None

@contextlib.contextmanager
def gpu_timer(run_name):
    start = time.time()
    try:
        yield
    finally:
        mins = (time.time() - start) / 60
        new = not os.path.exists(GPU_LOG)
        with open(GPU_LOG, "a", newline="") as f:
            w = csv.writer(f)
            if new:
                w.writerow(["run", "gpu", "start", "minutes"])
            gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
            w.writerow([run_name, gpu, datetime.datetime.fromtimestamp(start).isoformat(timespec="seconds"), round(mins, 1)])
        print(f"[{run_name}] {mins:.1f} menit GPU")

def load_names(yaml_path):
    n = yaml.safe_load(open(yaml_path))["names"]
    return list(n.values()) if isinstance(n, dict) else list(n)

def label_counts(root, names, split="*"):
    cnt = collections.Counter()
    for f in glob.glob(f"{root}/{split}/labels/*.txt"):
        cnt.update(names[int(l.split()[0])] for l in open(f) if l.strip())
    return cnt

def class_table(root, names):
    return (pd.DataFrame({s: label_counts(root, names, s) for s in ["train", "valid", "test"]})
              .fillna(0).astype(int).sort_values("train", ascending=False))

def write_yaml(path, root, names, train="train/images", val="valid/images", test="test/images"):
    yaml.safe_dump({"path": root, "train": train, "val": val, "test": test,
                    "names": dict(enumerate(names))}, open(path, "w"))

def merge_yolo(root, merge, drop):
    old = load_names(f"{root}/data.yaml")
    new_names = sorted({merge.get(n, n) for n in old} - set(drop))
    idx = {i: new_names.index(merge.get(n, n)) for i, n in enumerate(old) if merge.get(n, n) not in drop}
    before = label_counts(root, old)
    for lab in glob.glob(f"{root}/*/labels/*.txt"):
        rows = [l.split() for l in open(lab) if l.strip()]
        with open(lab, "w") as f:
            f.writelines(" ".join([str(idx[int(r[0])])] + r[1:]) + "\n" for r in rows if int(r[0]) in idx)
    d = yaml.safe_load(open(f"{root}/data.yaml"))
    d["names"], d["nc"] = new_names, len(new_names)
    yaml.safe_dump(d, open(f"{root}/data.yaml", "w"))
    expect = collections.Counter()
    for n, c in before.items():
        if merge.get(n, n) not in drop:
            expect[merge.get(n, n)] += c
    assert label_counts(root, new_names) == expect, "Merge kelas YOLO tidak konsisten!"
    return new_names

def merge_coco(root, merge, drop, names):
    cats = [{"id": i + 1, "name": n, "supercategory": "none"} for i, n in enumerate(names)]
    by_name = {c["name"]: c["id"] for c in cats}
    for p in glob.glob(f"{root}/*/_annotations.coco.json"):
        d = json.load(open(p))
        id2new = {c["id"]: merge.get(c["name"], c["name"]) for c in d["categories"]}
        d["annotations"] = [{**a, "category_id": by_name[id2new[a["category_id"]]]}
                            for a in d["annotations"] if id2new[a["category_id"]] not in drop]
        d["categories"] = cats
        json.dump(d, open(p, "w"))

## 3. Dataset dan merge kelas

In [4]:
from roboflow import Roboflow
rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))

ver = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION)
ds_yolo = ver.download("yolov8", location=f"{DATA}/train_yolo", overwrite=True)
ds_coco = ver.download("coco",   location=f"{DATA}/train_coco", overwrite=True)
C = ds_coco.location

raw = class_table(ds_yolo.location, load_names(f"{ds_yolo.location}/data.yaml"))
raw.to_csv(f"{ROOT}/class_counts_sebelum_merge.csv")
print(len(raw), "kelas sebelum merge"); display(raw)

NAMES = merge_yolo(ds_yolo.location, MERGE, DROP)
merge_coco(C, MERGE, DROP, NAMES)
cls = class_table(ds_yolo.location, NAMES)
cls.to_csv(f"{ROOT}/class_counts.csv")
print(len(NAMES), "kelas setelah merge"); display(cls)

TRAIN_YAML = f"{ds_yolo.location}/data_abs.yaml"
write_yaml(TRAIN_YAML, ds_yolo.location, NAMES)

json.dump({"MERGE": MERGE, "DROP": DROP, "NAMES": NAMES, "IMGSZ": IMGSZ,
           "dataset": [RF_WORKSPACE, RF_PROJECT, RF_VERSION]}, open(f"{ROOT}/train_config.json", "w"), indent=1)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /tmp/data/train_yolo in yolov8:: 100%|██████████| 1856/1856 [00:00<00:00, 6450.52it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings



Extracting Dataset Version Zip to /tmp/data/train_coco in coco:: 100%|██████████| 930/930 [00:00<00:00, 5596.22it/s]


74 kelas sebelum merge


,train,valid,test
wall-other-merged,756,315,650
table-merged,319,134,258
floor-other-merged,261,92,239
pan,225,91,34
chair,217,67,353
...,...,...,...
toothbrush,0,0,14
scissors,0,0,6
scissor,0,0,2
hair drier,0,0,2


72 kelas setelah merge


,train,valid,test
wall-other-merged,756,315,650
table-merged,319,134,258
floor-other-merged,261,92,239
pan,225,91,34
toy,217,77,41
...,...,...,...
flower,0,0,6
toothbrush,0,0,14
scissors,0,0,8
hair drier,0,0,2


## 4. Training YOLO (model utama)

In [5]:
from ultralytics import YOLO

NO_AUG = dict(hsv_h=0, hsv_s=0, hsv_v=0, degrees=0, translate=0, scale=0, shear=0,
              perspective=0, flipud=0, fliplr=0, mosaic=0, mixup=0, copy_paste=0)

CUSTOM_AUG = dict(
    hsv_v=0.6,          # cahaya redup / backlight (default 0.4)
    hsv_s=0.5,          # variasi warna lampu (kuning/putih)
    degrees=15,         # HP dipegang miring
    shear=5,
    perspective=0.0005, # sudut ekstrem (atas/bawah)
    copy_paste=0.3,     # objek saling menutupi
    mixup=0.1,
)

def train_yolo(weights, name, hours, **aug):
    run = f"{RUNS}/{name}"
    best = f"{run}/weights/best.pt"
    if os.path.exists(f"{run}/DONE"):
        print(f"[{name}] sudah selesai, skip"); return best
    last = f"{run}/weights/last.pt"
    with gpu_timer(name):
        if os.path.exists(last):
            YOLO(last).train(resume=True)
        else:
            YOLO(weights).train(data=TRAIN_YAML, imgsz=IMGSZ, time=hours, seed=SEED, deterministic=True,
                                device=0, workers=4, project=RUNS, name=name, exist_ok=True, **aug)
    open(f"{run}/DONE", "w").close()
    return best

YOLO_MAIN = {
    "YOLO11s-seg": stage("yolo11s_seg", train_yolo, "yolo11s-seg.pt", "yolo11s_seg", MAIN_HOURS),
    "YOLOv8s-seg": stage("yolov8s_seg", train_yolo, "yolov8s-seg.pt", "yolov8s_seg", MAIN_HOURS),
}

[yolo11s_seg] sudah selesai, skip
[yolov8s_seg] sudah selesai, skip


## 5. Training Mask R-CNN

In [6]:
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import torchvision.transforms.functional as TF
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as mask_util
from PIL import Image

NUM_CLASSES = len(NAMES) + 1

class CocoSeg(Dataset):
    def __init__(self, json_path, img_root, train=False, aug="default"):
        with contextlib.redirect_stdout(io.StringIO()):
            self.coco = COCO(json_path)
        self.root, self.train, self.aug = img_root, train, aug
        self.ids = sorted(self.coco.imgs)
        if train:
            self.ids = [i for i in self.ids if self._anns(i)]

    def _anns(self, iid):
        return [a for a in self.coco.loadAnns(self.coco.getAnnIds(imgIds=iid))
                if a.get("segmentation") and a["bbox"][2] >= 1 and a["bbox"][3] >= 1]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, k):
        iid = self.ids[k]
        info = self.coco.imgs[iid]
        img = TF.to_tensor(Image.open(f"{self.root}/{info['file_name']}").convert("RGB"))
        H, W = img.shape[-2:]
        anns = self._anns(iid)
        boxes = torch.tensor([[x, y, x + w, y + h] for x, y, w, h in (a["bbox"] for a in anns)],
                             dtype=torch.float32).reshape(-1, 4)
        masks = (torch.from_numpy(np.stack([self.coco.annToMask(a) for a in anns])) if anns
                 else torch.zeros((0, H, W), dtype=torch.uint8))
        labels = torch.tensor([a["category_id"] for a in anns], dtype=torch.int64)
        if self.train and self.aug != "none":
            img, masks, boxes, labels = augment(img, masks, boxes, labels, self.aug)
        return img, {"boxes": boxes, "labels": labels, "masks": masks, "image_id": iid}

def mask_boxes(masks):
    out = []
    for m in masks:
        ys, xs = torch.where(m > 0)
        out.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
    return torch.tensor(out, dtype=torch.float32).reshape(-1, 4)

def augment(img, masks, boxes, labels, mode):
    W = img.shape[-1]
    if torch.rand(1).item() < 0.5:
        img, masks = img.flip(-1), masks.flip(-1)
        boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
    if mode != "custom":
        return img, masks, boxes, labels
    u = lambda a, b: a + (b - a) * torch.rand(1).item()
    img = TF.adjust_brightness(img, u(0.4, 1.6))   # setara hsv_v=0.6
    img = TF.adjust_saturation(img, u(0.5, 1.5))   # setara hsv_s=0.5
    angle, shear = u(-15, 15), u(-5, 5)            # setara degrees=15, shear=5
    img = TF.affine(img, angle=angle, translate=[0, 0], scale=1.0, shear=[shear, 0.0]).clamp(0, 1)
    if len(masks):
        masks = (TF.affine(masks.float(), angle=angle, translate=[0, 0], scale=1.0, shear=[shear, 0.0],
                           interpolation=TF.InterpolationMode.NEAREST) > 0.5).to(torch.uint8)
        keep = masks.flatten(1).any(1)
        masks, labels = masks[keep], labels[keep]
        boxes = mask_boxes(masks)
    return img, masks, boxes, labels

def collate(batch):
    return tuple(zip(*batch))

def build_mrcnn(num_classes, pretrained=True):
    if pretrained:
        m = maskrcnn_resnet50_fpn_v2(weights="DEFAULT")
        m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, num_classes)
        m.roi_heads.mask_predictor = MaskRCNNPredictor(256, 256, num_classes)
    else:
        m = maskrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None, num_classes=num_classes)
    return m.to(DEV)

def load_mrcnn(path):
    m = build_mrcnn(NUM_CLASSES, pretrained=False)
    m.load_state_dict(torch.load(path, map_location=DEV))
    return m

@torch.no_grad()
def coco_eval_mrcnn(model, ds, quiet=False, per_class=None):
    model.eval()
    res = []
    for k in range(len(ds)):
        img, t = ds[k]
        out = model([img.to(DEV)])[0]
        for m, s, l in zip((out["masks"][:, 0] > 0.5).cpu().numpy(), out["scores"].tolist(), out["labels"].tolist()):
            rle = mask_util.encode(np.asfortranarray(m.astype(np.uint8)))
            rle["counts"] = rle["counts"].decode()
            res.append({"image_id": t["image_id"], "category_id": l, "segmentation": rle, "score": s})
    if not res:
        return 0.0, 0.0
    with contextlib.redirect_stdout(io.StringIO()) if quiet else contextlib.nullcontext():
        E = COCOeval(ds.coco, ds.coco.loadRes(res), "segm")
        E.evaluate(); E.accumulate(); E.summarize()
    if per_class is not None:
        prec = E.eval["precision"]
        for k, cid in enumerate(E.params.catIds):
            pk = prec[:, :, k, 0, -1]
            if (pk > -1).any():
                per_class[NAMES[cid - 1]] = float(pk[pk > -1].mean())
    return float(E.stats[0]), float(E.stats[1])

def train_mrcnn(name, hours, aug="default", base_lr=0.005, batch=4, warmup=200):
    out = f"{RUNS}/{name}"
    best = f"{out}/best.pt"
    if os.path.exists(f"{out}/DONE"):
        print(f"[{name}] sudah selesai, skip"); return best
    os.makedirs(out, exist_ok=True)
    torch.manual_seed(SEED)
    tr = CocoSeg(f"{C}/train/_annotations.coco.json", f"{C}/train", train=True, aug=aug)
    va = CocoSeg(f"{C}/valid/_annotations.coco.json", f"{C}/valid")
    dl = DataLoader(tr, batch_size=batch, shuffle=True, num_workers=2, collate_fn=collate)
    model = build_mrcnn(NUM_CLASSES)
    opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=base_lr, momentum=0.9, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    budget, t0, it, epoch, rows, best_map = hours * 3600, time.time(), 0, 0, [], -1.0
    with gpu_timer(name):
        while time.time() - t0 < budget:
            model.train()
            sums, n = collections.defaultdict(float), 0
            for imgs, tg in dl:
                decay = 0.1 if time.time() - t0 > 0.7 * budget else 1.0
                for g in opt.param_groups:
                    g["lr"] = base_lr * decay * min(1.0, (it + 1) / warmup)
                imgs = [i.to(DEV) for i in imgs]
                tg = [{k: v.to(DEV) if torch.is_tensor(v) else v for k, v in t.items()} for t in tg]
                with torch.autocast("cuda"):
                    losses = model(imgs, tg)
                loss = sum(losses.values())
                it += 1
                if not torch.isfinite(loss):
                    continue
                opt.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update()
                for k, v in losses.items():
                    sums[k] += v.item()
                n += 1
            epoch += 1
            m, m50 = coco_eval_mrcnn(model, va, quiet=True)
            rows.append({"epoch": epoch, "menit": round((time.time() - t0) / 60, 1), "lr": opt.param_groups[0]["lr"],
                         **{k: v / max(n, 1) for k, v in sums.items()}, "val_mask_mAP50-95": m, "val_mask_mAP50": m50})
            pd.DataFrame(rows).to_csv(f"{out}/results.csv", index=False)
            if m > best_map:
                best_map = m
                torch.save(model.state_dict(), best)
            print(rows[-1])
    open(f"{out}/DONE", "w").close()
    return best

MRCNN = stage("mask_rcnn_r50", train_mrcnn, "mask_rcnn_r50", MAIN_HOURS)

[mask_rcnn_r50] sudah selesai, skip


## 6. Eksperimen augmentasi (3 model)

Kondisi default = model utama di bagian 4 dan 5.

In [7]:
AUG_EXP = {
    "YOLO11s-seg": {
        "Tanpa augmentasi":  stage("yolo11s_aug_none",   train_yolo, "yolo11s-seg.pt", "yolo11s_aug_none",   MAIN_HOURS, **NO_AUG),
        "Default":           YOLO_MAIN["YOLO11s-seg"],
        "Rancangan sendiri": stage("yolo11s_aug_custom", train_yolo, "yolo11s-seg.pt", "yolo11s_aug_custom", MAIN_HOURS, **CUSTOM_AUG),
    },
    "YOLOv8s-seg": {
        "Tanpa augmentasi":  stage("yolov8s_aug_none",   train_yolo, "yolov8s-seg.pt", "yolov8s_aug_none",   MAIN_HOURS, **NO_AUG),
        "Default":           YOLO_MAIN["YOLOv8s-seg"],
        "Rancangan sendiri": stage("yolov8s_aug_custom", train_yolo, "yolov8s-seg.pt", "yolov8s_aug_custom", MAIN_HOURS, **CUSTOM_AUG),
    },
    "Mask R-CNN R50-FPN v2": {
        "Tanpa augmentasi":  stage("mask_rcnn_aug_none",   train_mrcnn, "mask_rcnn_aug_none",   MAIN_HOURS, aug="none"),
        "Default":           MRCNN,
        "Rancangan sendiri": stage("mask_rcnn_aug_custom", train_mrcnn, "mask_rcnn_aug_custom", MAIN_HOURS, aug="custom"),
    },
}

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/tmp/data/train_yolo/data_abs.yaml, degrees=0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0, flipud=0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0, hsv_s=0, hsv_v=0, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=0, multi_scale=0.0, name=yolo11s_aug_none, nbs=64, nms=None, opset=None, optimize=False, optim

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/185      6.41G     0.9076      2.141      3.069      1.218          0        164        640: 100% ━━━━━━━━━━━━ 25/25 3.7it/s 6.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.7it/s 2.2s
                   all        166       2021      0.429      0.218      0.222      0.183      0.415      0.219      0.221      0.154

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      3/246      6.43G     0.7689      1.726      2.267      1.117          0        143        640: 100% ━━━━━━━━━━━━ 25/25 3.7it/s 6.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.5it/s 2.4s
                   all        166       2021      0.538      0.242      0.241      0.192      0.534      0.241      0.242      0.166

      Epoch    GPU

100%|██████████| 177M/177M [00:00<00:00, 225MB/s]


{'epoch': 1, 'menit': 2.2, 'lr': 0.0025, 'loss_classifier': 1.5214502918720245, 'loss_box_reg': 0.6470967456698418, 'loss_mask': 0.6895384413003921, 'loss_objectness': 0.15662263985723257, 'loss_rpn_box_reg': 0.09505751673132182, 'val_mask_mAP50-95': 0.007133146137890869, 'val_mask_mAP50': 0.013446285003573281}
{'epoch': 2, 'menit': 4.6, 'lr': 0.005, 'loss_classifier': 0.7785190659761428, 'loss_box_reg': 0.5735863281786442, 'loss_mask': 0.5373654565215111, 'loss_objectness': 0.054064483065158125, 'loss_rpn_box_reg': 0.07317167541012168, 'val_mask_mAP50-95': 0.05184113443662369, 'val_mask_mAP50': 0.08464416203433321}
{'epoch': 3, 'menit': 6.9, 'lr': 0.005, 'loss_classifier': 0.6056160612404347, 'loss_box_reg': 0.4447550056874752, 'loss_mask': 0.42485194325447084, 'loss_objectness': 0.04078591932542622, 'loss_rpn_box_reg': 0.06362879956141114, 'val_mask_mAP50-95': 0.08302054746936387, 'val_mask_mAP50': 0.13604568907158937}
{'epoch': 4, 'menit': 9.3, 'lr': 0.005, 'loss_classifier': 0.4973

indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:37.)
indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:37.)


... (47 outputs left out)

indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:37.)
indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:37.)


{'epoch': 25, 'menit': 58.0, 'lr': 0.0005, 'loss_classifier': 0.2135892955213785, 'loss_box_reg': 0.1873421214520931, 'loss_mask': 0.20057315215468408, 'loss_objectness': 0.01704074294073507, 'loss_rpn_box_reg': 0.04194843261502683, 'val_mask_mAP50-95': 0.1893036337550112, 'val_mask_mAP50': 0.27569315213428863}


indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:37.)
indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:37.)


{'epoch': 26, 'menit': 60.2, 'lr': 0.0005, 'loss_classifier': 0.2052570355683565, 'loss_box_reg': 0.1819223017990589, 'loss_mask': 0.19662255480885504, 'loss_objectness': 0.03324768126476556, 'loss_rpn_box_reg': 0.06185628933832049, 'val_mask_mAP50-95': 0.1864243433750173, 'val_mask_mAP50': 0.2740292786192658}
[mask_rcnn_aug_custom] 60.2 menit GPU


## 7. Ringkasan

In [8]:
print("Waktu GPU:"); display(pd.read_csv(GPU_LOG)) if os.path.exists(GPU_LOG) else print("-")
print(open(ERR_LOG).read() if os.path.exists(ERR_LOG) else "Tidak ada tahap yang gagal.")

Waktu GPU:


,run,gpu,start,minutes
0,yolo11s_seg,Tesla T4,2026-11-17T15:12:12,27.9
1,yolov8s_seg,Tesla T4,2026-11-17T15:40:05,29.7
2,mask_rcnn_r50,Tesla T4,2026-11-17T16:09:50,61.2
3,aug_none,Tesla T4,2026-11-17T17:11:05,12.5
4,aug_default,Tesla T4,2026-11-17T17:23:35,28.5
5,aug_custom,Tesla T4,2026-11-17T17:52:06,30.2
6,yolo11s_aug_none,Tesla T4,2026-11-24T08:58:30,20.3
7,yolo11s_aug_custom,Tesla T4,2026-11-24T09:18:46,34.8
8,yolov8s_aug_none,Tesla T4,2026-11-24T09:53:32,18.2
9,yolov8s_aug_custom,Tesla T4,2026-11-24T10:11:41,53.5


Tidak ada tahap yang gagal.
